# 재활용품 5종 인식 실험 보고서
**YOLO26n · ResNet18 · Grad-CAM · DeepLabV3+** | 2026-11-13

> **질문:** 색상/조명 증강과 객체 분할이 혼잡한 폐기물 장면의 탐지·재질 분류를 실제로 개선할까?  
> **결론:** ROI 분류 개선과 분할 성능은 확인했지만, 최종 End-to-End에서는 **YOLO26n RAW 단독**이 가장 단순하고 유리했다.

원본: `261113_zerowaste_reuse_experiments.ipynb` (38개 셀). 이 문서는 기존 결과를 요약하는 **실행 가능한 분석용 노트북**이며, 이미 완료한 GPU 학습을 자동으로 다시 수행하지 않는다.


## 1. 데이터와 평가 설계

| 구분 | Train | Validation | Test |
|---|---:|---:|---:|
| 이미지 | 2,800 | 600 | 591 |
| GT 객체 ROI | 6,214 | 1,365 | — |
| Polygon 객체 | 2,270 | 456 | — |

**5개 클래스:** styrofoam, vinyl, brown_glass, green_glass, clear_glass.  
기존 TAR와 `.pt` 체크포인트를 재사용해 실험마다 새 데이터 분할이나 사전학습을 반복하지 않았다.

**측정 단위 주의:** ROI 분류 *Macro-F1* ≠ 검출 *End-to-End F1* ≠ 분할 *mIoU*. 검출 TP는 **동일 클래스 + Box IoU ≥ 0.5**를 요구한다. 일부 이미지에서 다른 클래스의 GT가 빠져 있어 정확한 탐지도 FP로 잡힐 수 있다.


In [ ]:
# 결과만 다시 조회하는 셀: Colab에서 실행
from pathlib import Path
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # 로컬 Jupyter에서는 Drive를 마운트하거나 REPORT_ROOT를 직접 지정

REPORT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/AI응용/"
    "261113_zerowaste_reuse_experiments/experiments_v3"
)

def read_result(relative_path):
    path = REPORT_ROOT / relative_path
    if not path.exists():
        print("파일 없음:", path)
        return None
    return pd.read_csv(path)

print("분석 결과 폴더:", REPORT_ROOT)


## 2. 색상/밝기 증강: 검증 성능은 일부 개선, 통합 성능은 미미

**왜?** 유리병은 조명·반사·라벨 때문에 RGB/HSV 임계값으로 잘 분리되지 않았다. 색 자체가 정답인 문제라서 큰 Hue 변경은 오히려 물체 색상 단서를 훼손할 수 있어, **V(밝기)만 변화**하는 실험과 약한 HSV 증강을 비교했다.

- **YOLO 증강 실험의 Validation Macro-F1:** control **70.34%**, brightness **71.63%** (**+1.29%p**), mild HSV **71.18%**.
- 그러나 별도 End-to-End 검증에서는 **RAW 71.03% vs brightness 70.90%**. 녹색 병은 brightness가 유리했지만 투명 병·비닐은 약해졌다.
- **트레이드오프:** 다양한 조명에 대한 적응 가능성 ↔ 클래스 간 색상 차이 약화, TP 감소, 실험 설정에 따른 성능 차이.


In [ ]:
# 원본 노트북의 핵심 실험 설정 (학습 재실행은 호출할 때만)
YOLO_AUGMENTATIONS = {
    "control":    {"hsv_h": 0.000, "hsv_s": 0.0, "hsv_v": 0.00},
    "brightness": {"hsv_h": 0.000, "hsv_s": 0.0, "hsv_v": 0.15},
    "mild_hsv":   {"hsv_h": 0.005, "hsv_s": 0.1, "hsv_v": 0.15},
}

def train_yolo_variant(start_weights, data_yaml, variant, output_folder):
    from ultralytics import YOLO
    cfg = YOLO_AUGMENTATIONS[variant]
    return YOLO(str(start_weights)).train(
        data=str(data_yaml), project=str(output_folder), name=variant,
        epochs=12, imgsz=640, batch=16, device=0,
        optimizer="AdamW", lr0=5e-4, lrf=0.1,
        seed=42, deterministic=True, **cfg
    )

aug = read_result("augmentation_ablation/augmentation_summary.csv")
if aug is not None:
    display(aug)
else:
    print("학습은 반복하지 않고, 아래 요약 수치를 참고하세요.")


## 3. ResNet18: ROI 분류는 개선, 통합 파이프라인의 이익은 제한

**왜?** YOLO의 Box 분류와 별도로 객체 crop을 ResNet18에 넣어 형태·색·라벨 패턴과 손상/오염/겹침/투명도를 다섯 헤드로 학습·분석했다.

- **동일한 Validation GT ROI 1,365개:** RAW 체크포인트 **93.36% → Kornia 95.15% Macro-F1**, **+1.79%p**. 오분류 **75 → 54개**.
- **Dropout p=0 / 0.2 / 0.4:** 클래스 F1 **95.10 / 95.48 / 95.61%**, 5-head 평균 **75.32 / 75.09 / 75.19%**. 큰 우열은 없다.
- **트레이드오프:** ROI에서는 분류를 잘하지만, 추가 추론 비용이 들고 앞단의 누락 Box(FN)는 복구할 수 없다. Dropout 실험의 속성 라벨은 JSON에서 재구성했으므로 과거 5-head 점수와 직접 비교하면 안 된다(`dirtiness_f1=0.5`).


In [ ]:
# 실제 모델 구조의 요약: ImageNet ResNet18 + 공유 Backbone + 5개 Head
import torch.nn as nn
from torchvision.models import resnet18

HEAD_DIMS = {"class_id": 5, "damage": 2, "dirtiness": 2,
             "cover": 2, "transparency": 2}

class MultiTaskResNet18(nn.Module):
    def __init__(self, p=0.2):
        super().__init__()
        self.backbone = resnet18(weights=None)
        dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.heads = nn.ModuleDict({
            h: nn.Sequential(nn.Dropout(p), nn.Linear(dim, n))
            for h, n in HEAD_DIMS.items()
        })

    def forward(self, x):
        z = self.backbone(x)
        return {h: layer(z) for h, layer in self.heads.items()}

roi_comparison = read_result("evaluation/raw_vs_kornia_same_roi/model_comparison_summary.csv")
if roi_comparison is not None:
    display(roi_comparison)


## 4. Grad-CAM에서 실제 Polygon Segmentation으로

**왜?** Grad-CAM은 분류에 중요한 위치를 보여주지만 정확한 객체 외곽선은 보장하지 않는다. 특히 병뚜껑·상품 라벨·배경에 집중하여 `CAM ≥ 0.45` 마스크가 누락·과잉 포함을 만들었다.

AI Hub JSON의 객체 라벨을 재확인해 **Polygon 2,726개(Train 2,270 / Val 456)** 를 발견했다. 이들을 직접 `cv2.fillPoly`로 정답 픽셀 마스크로 변환하고 **DeepLabV3+ (ResNet50)** 를 학습했다.

- **Validation Polygon ROI Foreground mIoU 79.43%** (Epoch 1: 68.05% → Epoch 8: 79.43%, **+11.38%p**).
- **클래스별 IoU:** styrofoam **96.02%**, vinyl **82.99%**, brown **67.57%**, green **73.48%**, clear **77.12%**.
- **트레이드오프:** 정확한 객체 영역 학습이 가능하지만 Polygon이 적은 갈색 병은 취약하다. `ignore_index=255`로 제외된 패딩·중첩 영역의 오예측은 mIoU에 포함되지 않는다.


In [ ]:
# 원본 셀 14~15에서 사용한 핵심 마스크/학습 원리
import cv2
import numpy as np

def polygon_to_mask(points_xy, height, width, class_id):
    mask = np.zeros((height, width), dtype=np.uint8)
    # class 0은 배경, 1~5는 재활용품; BOX 인접 영역과 패딩은 255(ignore) 처리
    cv2.fillPoly(mask, [np.rint(points_xy).astype(np.int32)], class_id + 1)
    return mask

def build_deeplabv3plus():
    import segmentation_models_pytorch as smp
    return smp.DeepLabV3Plus(
        encoder_name="resnet50", encoder_weights="imagenet",
        encoder_output_stride=16, in_channels=3, classes=6,
        activation=None
    )

# 모델 학습 시: torch.nn.CrossEntropyLoss(ignore_index=255)
iou = read_result(
    "deeplabv3plus_polygon/deeplabv3plus_resnet50/final_class_iou.csv"
)
if iou is not None:
    display(iou)


## 5. 분할 마스크가 분류를 개선했는가?

**동일한 456개 Polygon Validation ROI에 고정된 Kornia ResNet18 사용:**

| 입력 | Macro-F1 | RAW 대비 |
|---|---:|---:|
| 원본 RAW ROI | **95.02%** | 기준 |
| DeepLab 예측 마스크로 배경 제거 | 89.54% | **−5.48%p** |
| GT Polygon 마스크로 배경 제거 | 92.20% | **−2.82%p** |

**원인 가설:** 원본 배경과 명암 정보를 보고 학습한 분류기에 배경을 갑자기 회색으로 바꾸면 입력 분포가 변한다. *마스크의 정확성만 높여서는 충분하지 않았다.*

**대응:** Train에서 RAW와 GTMask를 50:50으로 섞어 3 epoch 파인튜닝 → 예측 마스크 F1 **89.54 → 92.32% (+2.78%p)**. 하지만 같은 모델의 RAW 입력 **94.76%**, 기존 Kornia+RAW **95.02%**보다 낮다. 따라서 **분할 모델을 최종 검출 파이프라인에 넣지 않았다.**


In [ ]:
# 핵심 마스킹 코드: 정답 클래스 정보를 사용하지 않고 '모든 비배경 예측' 채택
def neutral_background(bgr, predicted_seg):
    fg = predicted_seg != 0  # 모델 예측 1~5를 모두 전경 후보로
    output = bgr.copy()
    output[~fg] = (114, 114, 114)  # 원본 배경 대신 중립 회색
    return output

paired = read_result("segmentation_classification/classification_summary.csv")
mixed = read_result(
    "segmentation_classification/mask_aware_finetune/classification_comparison.csv"
)
if paired is not None: display(paired[["variant", "macro_f1", "accuracy"]])
if mixed is not None:
    display(mixed.pivot(index="model", columns="validation_input", values="macro_f1"))


## 6. 최종 통합 평가 및 모델 선택

검출 판단은 **Confidence ≥ 0.286235 / NMS IoU 0.45 / GT IoU ≥ 0.5**, Validation 600장으로 고정했다.

| 구성 | Val Macro-F1 | Val Micro-F1 |
|---|---:|---:|
| **YOLO26n RAW** | **71.03%** | 71.85% |
| YOLO26n Brightness | 70.90% | **71.89%** |
| Brightness + Kornia ResNet18(클래스 재판정) | 70.67% | 71.69% |

**결정:** Macro-F1, 재현율, 운영 단순성을 고려해 **YOLO RAW 단독** 채택. Brightness는 FP를 줄였으나 TP도 감소했고, ResNet 재판정은 평균적으로 개선되지 않았다.

**최종 Test 591장 (고정된 RAW 모델):** Macro-F1 **65.68%**, Micro-F1 **63.20%** (TP **869**, FP **634**, FN **378**). Validation Macro-F1보다 **5.34%p 낮음**. 비닐 F1 **49.03%**, 투명 유리병 **80.28%**. 비닐 FP **312개**로 전체 FP의 **49.2%**.

**한계:** 특정 종류만 표기된 이미지 때문에 FP 과대계상 가능성이 있고, Test는 이전 연구에서 이미 조회한 적이 있어 엄밀한 최초 블라인드 테스트는 아니다.


In [ ]:
# 원본 셀 18~19에서 산출한 결과를 재확인하고 시각화
import matplotlib.pyplot as plt

val = read_result("evaluation/end_to_end_validation/end_to_end_summary.csv")
test = read_result("evaluation/final_test/test_summary.csv")
class_test = read_result("evaluation/final_test/test_class_metrics.csv")

if val is not None:
    display(val[["model", "macro_f1", "micro_f1", "TP", "FP", "FN"]])
if test is not None:
    display(test[["model", "images", "macro_f1", "micro_f1", "TP", "FP", "FN"]])
if class_test is not None:
    display(class_test[["class", "precision", "recall", "f1"]])
    ax = class_test.plot.bar(x="class", y="f1", legend=False, figsize=(8, 3.2))
    ax.set_ylim(0, 1)
    ax.set_ylabel("Test F1")
    ax.set_title("Final Test — YOLO26n RAW")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


## 결론 · 얻은 교훈

**성공:** 원본 산출물을 재사용해 ① 밝기·색상 증강 ② ResNet18 ROI 분류 ③ Grad-CAM 오류 해석 ④ Polygon 기반 DeepLabV3+ ⑤ Mask-aware 재학습 ⑥ 실제 End-to-End까지 **같은 데이터 분할과 정량 지표**로 비교했다.

**채택하지 않은 방법도 근거가 생김:** 조명 증강은 단일 실험에서는 개선됐지만 최종 통합 이득이 작았고, Mask-aware 학습도 예측 마스크 성능을 회복했으나 RAW 기반 분류기를 이기지 못했다. 모델을 더 쌓는 것보다 **GT 완전성, 데이터 클래스 분포, 검출 누락(FN)과 FP 분석**이 다음 성능 향상에 중요하다.

*원본 학습 스크립트와 저장된 `.pt`·CSV·시각화는 위 `experiments_v3` 아래에 보존. 본 문서는 결과 해설 및 재조회용으로 작성됨.*
